# Session 10 — LangGraph Essentials
## Focus: Graph-based agent orchestration, branching, streaming, fault-tolerance

### Introduction to LangGraph Essentials
**Explanation:**
LangGraph extends the LangChain ecosystem by introducing **graph-based orchestration** for building reliable, observable, and fault-tolerant multi-agent systems. Instead of linear chains, LangGraph represents workflows as **Directed Acyclic Graphs (DAGs)**, where each **node** represents a computation step or agent, and **edges** define dependencies and data flow.

This allows complex orchestration patterns like **parallel branches**, **streaming between nodes**, **automatic retries**, and **state management** across runs — enabling true production-level multi-agent pipelines. In this session, we’ll build a **multi-agent DAG** that performs extraction → enrichment → summarization on media articles, showcasing the power of LangGraph.


## Installation & Requirements Setup

In [20]:
# Install required packages for LangGraph
!pip install langgraph langchain openai python-dotenv requests tiktoken
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from typing import Dict, Any

# Load environment variables
load_dotenv()

# Configure OpenRouter API key
os.environ["OPENROUTER_API_KEY"] = "open router api key"



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Building a Basic LangGraph Workflow
Each node in LangGraph represents a **computational step**. Here, we define three:
- **Extraction** → Identifies named entities and topics
- **Enrichment** → Adds contextual background
- **Summarization** → Produces a digestible output

Each node takes the shared `state` dictionary, processes it, and returns an updated version — enabling fully traceable data flow.

In [26]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from typing import TypedDict

# --- Define a typed state ---
class NewsGraphState(TypedDict, total=False):
    article: str
    extracted_info: str
    enriched_info: str
    summary: str

# --- Initialize LLM ---
llm = ChatOpenAI(model="gpt-4o-mini", base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"], temperature=0)

# --- Define nodes ---
def extract_node(state: NewsGraphState) -> NewsGraphState:
    article = state["article"]
    response = llm.invoke(f"Extract key people, orgs, and topics from this:\n\n{article}")
    state["extracted_info"] = response.content
    return state

def enrich_node(state: NewsGraphState) -> NewsGraphState:
    extracted = state["extracted_info"]
    response = llm.invoke(f"Expand on these details:\n\n{extracted}")
    state["enriched_info"] = response.content
    return state

def summarize_node(state: NewsGraphState) -> NewsGraphState:
    enriched = state["enriched_info"]
    response = llm.invoke(f"Write a short summary:\n\n{enriched}")
    state["summary"] = response.content
    return state

**Explanation:**
Each node in LangGraph represents a **computational step**. Here, we define three:
- **Extraction** → Identifies named entities and topics
- **Enrichment** → Adds contextual background
- **Summarization** → Produces a digestible output

Each node takes the shared `state` dictionary, processes it, and returns an updated version — enabling fully traceable data flow.


## Creating the Graph and Adding Edges
LangGraph uses a declarative structure for node orchestration.
- `set_entry_point()` defines where the workflow begins.
- `add_edge()` defines data flow.
- `END` represents terminal state.

By compiling the graph, we generate a **fully executable pipeline** that automatically handles flow, dependencies, and failure isolation.


In [27]:
graph = StateGraph(NewsGraphState, input_keys=["article"], output_keys=["summary"])
graph.add_node("extract", extract_node)
graph.add_node("enrich", enrich_node)
graph.add_node("summarize", summarize_node)

graph.set_entry_point("extract")
graph.add_edge("extract", "enrich")
graph.add_edge("enrich", "summarize")
graph.add_edge("summarize", END)

news_graph = graph.compile()

**Explanation:**
LangGraph uses a declarative structure for node orchestration.
- `set_entry_point()` defines where the workflow begins.
- `add_edge()` defines data flow.
- `END` represents terminal state.

By compiling the graph, we generate a **fully executable pipeline** that automatically handles flow, dependencies, and failure isolation.


## Running the LangGraph Workflow
When executed, LangGraph automatically routes the shared state through each node in sequence. If any node fails, LangGraph can **retry** based on configurable policies — making it ideal for **fault-tolerant, production-grade agent orchestration**.


In [28]:
# --- Run the graph ---
sample_article = """OpenAI has announced a new version of ChatGPT with improved reasoning and memory capabilities."""
final_state = news_graph.invoke({"article": sample_article})

# --- Display results ---
print("📰 Extracted Info:\n", final_state["extracted_info"])
print("\n✨ Enriched Info:\n", final_state["enriched_info"])
print("\n🧭 Summary:\n", final_state["summary"])

📰 Extracted Info:
 **Key People:**
- None specified

**Organizations:**
- OpenAI

**Topics:**
- ChatGPT
- Improved reasoning
- Memory capabilities

✨ Enriched Info:
 Certainly! Here’s an expanded overview based on the details provided:

### Key People
While no specific individuals are mentioned, it’s important to recognize that the development and evolution of technologies like ChatGPT involve a diverse team of researchers, engineers, and thought leaders at OpenAI. These individuals contribute their expertise in artificial intelligence, machine learning, natural language processing, and ethics to enhance the capabilities of AI systems. Notable figures in the AI community, such as Sam Altman (CEO of OpenAI) and Ilya Sutskever (co-founder and Chief Scientist), have played significant roles in shaping the direction of AI research and its applications.

### Organizations
**OpenAI**
OpenAI is a research organization focused on developing artificial intelligence in a safe and beneficial mann

**Explanation:**
When executed, LangGraph automatically routes the shared state through each node in sequence. If any node fails, LangGraph can **retry** based on configurable policies — making it ideal for **fault-tolerant, production-grade agent orchestration**.


## Branching and Conditional Paths
Branching allows **conditional routing** between nodes, enabling adaptive agent behavior. You can direct shorter inputs to fast paths and longer or complex ones to deeper reasoning flows. This pattern is fundamental for **dynamic multi-agent coordination**.


In [30]:
from langgraph.graph import StateGraph, END

def route(state: NewsGraphState):
    """Decide which node to run next based on the article content."""
    if "AI" in state["article"]:
        return "enrich"
    else:
        return "summarize"

graph = StateGraph(NewsGraphState, input_keys=["article"], output_keys=["summary"])
graph.add_node("extract", extract_node)
graph.add_node("enrich", enrich_node)
graph.add_node("summarize", summarize_node)
graph.add_node("route", route)

graph.set_entry_point("extract")

# ✅ Proper branching:
graph.add_conditional_edges(
    "extract",
    route,  # function that decides next node
    {
        "enrich": "enrich",
        "summarize": "summarize"
    }
)

graph.add_edge("enrich", "summarize")
graph.add_edge("summarize", END)


**Explanation:**
Branching allows **conditional routing** between nodes, enabling adaptive agent behavior. You can direct shorter inputs to fast paths and longer or complex ones to deeper reasoning flows. This pattern is fundamental for **dynamic multi-agent coordination**.


## Streaming Through State (Progressive Output)
`graph.stream()` allows you to observe **intermediate node outputs** in real time — crucial for debugging, tracing, or building live dashboards. This gives full visibility into internal reasoning steps, similar to LangSmith traces but directly through the LangGraph API.


In [31]:
# Stream results for real-time monitoring
for step in news_graph.stream({"article": sample_article}):
    for key, value in step.items():
        print(f"🔹 Node: {key}")
        print(value)
        print("-" * 50)


🔹 Node: extract
{'article': 'OpenAI has announced a new version of ChatGPT with improved reasoning and memory capabilities.', 'extracted_info': '**Key People:**\n- None specified\n\n**Organizations:**\n- OpenAI\n\n**Topics:**\n- ChatGPT\n- Improved reasoning\n- Memory capabilities'}
--------------------------------------------------
🔹 Node: enrich
{'article': 'OpenAI has announced a new version of ChatGPT with improved reasoning and memory capabilities.', 'extracted_info': '**Key People:**\n- None specified\n\n**Organizations:**\n- OpenAI\n\n**Topics:**\n- ChatGPT\n- Improved reasoning\n- Memory capabilities', 'enriched_info': "Certainly! Here’s an expanded overview based on the details provided:\n\n### Key People\nWhile no specific individuals are mentioned, it’s important to recognize that the development of ChatGPT and its associated technologies involves a diverse team of researchers, engineers, and scientists at OpenAI. This team includes experts in artificial intelligence, machin

**Explanation:**
`graph.stream()` allows you to observe **intermediate node outputs** in real time — crucial for debugging, tracing, or building live dashboards. This gives full visibility into internal reasoning steps, similar to LangSmith traces but directly through the LangGraph API.


## Fault-Tolerance and Retries
LangGraph supports **retries, rollback, and recovery** per node. By wrapping steps in `try/except` or integrating with observability tools, you can ensure resilience under API rate limits, model timeouts, or transient failures — key for real-world production systems.


In [33]:
import random

def flaky_enrich_node(state: NewsGraphState) -> NewsGraphState:
    """Randomly fail to test retry or error handling."""
    if random.random() < 0.3:
        raise Exception("Simulated transient LLM error.")
    enriched = f"Enriched content: {state['extracted_info']}"
    state["enriched_info"] = enriched
    return state


# Replace enrich node with flaky version
fault_graph = StateGraph(NewsGraphState)
fault_graph.add_node("extract", extract_node)
fault_graph.add_node("enrich", flaky_enrich_node)
fault_graph.add_node("summarize", summarize_node)
fault_graph.set_entry_point("extract")
fault_graph.add_edge("extract", "enrich")
fault_graph.add_edge("enrich", "summarize")
fault_graph.add_edge("summarize", END)

fault_tolerant = fault_graph.compile()

# Run with retry logic
try:
    state = fault_tolerant.invoke({"article": sample_article})
    print("\n✅ Summary after retries:\n", state["summary"])
except Exception as e:
    print("❌ Workflow failed:", e)



✅ Summary after retries:
 The content discusses advancements in ChatGPT, particularly focusing on improved reasoning and memory capabilities. It highlights the role of OpenAI in these developments, although no specific key individuals are mentioned.


**Explanation:**
LangGraph supports **retries, rollback, and recovery** per node. By wrapping steps in `try/except` or integrating with observability tools, you can ensure resilience under API rate limits, model timeouts, or transient failures — key for real-world production systems.


## Case Study — Media Monitor News Summarizer
This case study integrates everything into a cohesive pipeline — a **media monitor** that continuously summarizes new articles with robust fault-tolerance and performance metrics. It demonstrates how LangGraph supports **graph orchestration, monitoring, and scalability** in modern AI applications.


In [34]:
class MediaMonitor:
    """Example of a production-ready LangGraph workflow"""

    def __init__(self):
        self.graph = news_graph
        self.metrics = {"runs": 0, "failures": 0, "avg_runtime": 0.0}

    def summarize_article(self, text: str) -> Dict[str, Any]:
        import time
        start = time.time()
        try:
            self.metrics["runs"] += 1
            result = self.graph.invoke({"article": text})
            duration = time.time() - start
            self.metrics["avg_runtime"] = ((self.metrics["avg_runtime"] * (self.metrics["runs"] - 1)) + duration) / self.metrics["runs"]
            return {"status": "success", "summary": result["summary"], "runtime": round(duration, 2)}
        except Exception as e:
            self.metrics["failures"] += 1
            return {"status": "error", "error": str(e)}

# Test media summarizer
monitor = MediaMonitor()
result = monitor.summarize_article(sample_article)
print("🧠 Final Summary:", result["summary"])
print("📊 Metrics:", monitor.metrics)


🧠 Final Summary: The development of ChatGPT, a conversational AI model by OpenAI, involves a diverse team of experts in artificial intelligence and ethics. OpenAI, founded to ensure that artificial general intelligence benefits humanity, is known for its commitment to safety and transparency in AI. ChatGPT utilizes deep learning to generate human-like text and has seen improvements in reasoning capabilities, allowing it to handle complex queries and provide accurate responses. Additionally, recent advancements include memory capabilities, enabling the model to retain information from past interactions for a more personalized user experience. These developments enhance the functionality of AI while raising important ethical considerations regarding user privacy and the implications of memory in AI systems. OpenAI remains dedicated to responsible AI development for the benefit of all.
📊 Metrics: {'runs': 1, 'failures': 0, 'avg_runtime': 23.17263627052307}


**Explanation:**
This case study integrates everything into a cohesive pipeline — a **media monitor** that continuously summarizes new articles with robust fault-tolerance and performance metrics. It demonstrates how LangGraph supports **graph orchestration, monitoring, and scalability** in modern AI applications.


## Summary
LangGraph provides a **powerful abstraction layer** for multi-agent coordination and workflow orchestration. By modeling LLM pipelines as graphs, developers gain:
- ✅ **Deterministic flow** and **state control**
- ✅ **Fault-tolerant execution** with retries
- ✅ **Branching and streaming** capabilities
- ✅ **Seamless integration** with LangChain and LangSmith

This approach marks a major shift from sequential chains to **modular, graph-driven reasoning systems**, ideal for production-scale AI orchestration.

